In [1]:
import sys
!{sys.executable} -m ensurepip --upgrade
# Install for pip if needed

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /tmp/tmp4e3l6sko


In [2]:
!{sys.executable} -m pip install --upgrade pip
# If system needs to download pip you must then run this to upgrade pip

Defaulting to user installation because normal site-packages is not writeable


In [3]:
!{sys.executable} -m pip install torch torchvision scikit-learn
# Once pip is upgraded run this to install torchvision which is needed for EfficientNet

Defaulting to user installation because normal site-packages is not writeable


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from PIL import Image
from torch.utils.data import Dataset

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
#I had to look up how to use this properly, but this forces your GPU to be used if you have but still allows users with no GPU's to run it

cpu


In [6]:
csv_path = "HAM10000_metadata.csv"
img_1 = "HAM10000_images_part_1"
img_2 = "HAM10000_images_part_2"

df = pd.read_csv(csv_path)
print(df.head())

     lesion_id      image_id   dx dx_type   age   sex localization  \
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp   
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp   
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp   
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp   
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear   

        dataset  
0  vidir_modern  
1  vidir_modern  
2  vidir_modern  
3  vidir_modern  
4  vidir_modern  


In [7]:
label_map = {label: idx for idx, label in enumerate(df['dx'].unique())}
df['label'] = df['dx'].map(label_map)

print(label_map)
# This converts the labels from things like mel, bkl, and df to numbers which are easier to compute

{'bkl': 0, 'nv': 1, 'df': 2, 'mel': 3, 'vasc': 4, 'bcc': 5, 'akiec': 6}


In [8]:
class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, img_dir1, img_dir2, transform=None):
        self.df = dataframe
        self.img_dir1 = img_dir1
        self.img_dir2 = img_dir2
        self.transform = transform
        self.classes = sorted(dataframe['dx'].unique())

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['image_id']
        label = self.df.iloc[idx]['label']

        # Look through both folders for images
        img_path = os.path.join(self.img_dir1, img_id + ".jpg")
        if not os.path.exists(img_path):
            img_path = os.path.join(self.img_dir2, img_id + ".jpg")

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [9]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
#Transforming the images like this helps make sure the orientation of the image doesn't affect how the model looks at the lesions, and instead the model only cares about what the lesion actually looks like

In [10]:
dataset = HAM10000Dataset(df, img_1, img_2, transform=transform)

print("Dataset size:", len(dataset))

Dataset size: 10015


In [11]:
from torch.utils.data import random_split, DataLoader

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [12]:
model = models.efficientnet_b0(pretrained=True)

# Freeze feature layers
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier
num_classes = len(dataset.classes)

model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

model = model.to(device)

/opt/miniconda3/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/miniconda3/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [14]:
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")

Epoch 1, Loss: 231.8875
Epoch 2, Loss: 190.1737
Epoch 3, Loss: 178.3161
Epoch 4, Loss: 172.5464
Epoch 5, Loss: 165.5878


In [15]:
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=dataset.classes))



Accuracy: 0.7683474787818273
              precision    recall  f1-score   support

       akiec       0.52      0.50      0.51       218
         bcc       0.86      0.93      0.89      1340
         bkl       0.50      0.07      0.13        27
          df       0.48      0.38      0.42       204
         mel       0.70      0.63      0.67        30
          nv       0.60      0.50      0.55       110
        vasc       0.53      0.35      0.42        74

    accuracy                           0.77      2003
   macro avg       0.60      0.48      0.51      2003
weighted avg       0.75      0.77      0.75      2003



The loss is decreasing every EPOCH which is good, however it doesn't handle rare cases very well